In [1]:

!pip install kaggle torch torchaudio librosa numpy pandas scikit-learn transformers xgboost joblib -q

In [2]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# ============================================================
# CELL 2 — Download datasets
# 1. ICBHI Respiratory Sound DB  → wheeze / crackle / normal
# 2. COVID-19 Cough Audio        → cough classification
# 3. Lung Dataset                → broader lung sounds
# ============================================================
!kaggle datasets download -d vbookshelf/respiratory-sound-database
!unzip -o respiratory-sound-database.zip -d respiratory_db

!kaggle datasets download -d andrewmvd/covid19-cough-audio-classification
!unzip -o covid19-cough-audio-classification.zip -d cough_db

!kaggle datasets download -d arashnic/lung-dataset
!unzip -o lung-dataset.zip -d lung_db

Streaming output truncated to the last 5000 lines.
  inflating: cough_db/eab8daf8-7570-474c-8706-a8aa38cd7680.webm  
  inflating: cough_db/eabb0dbf-be86-4c8d-922c-8bbc7f6557f1.json  
  inflating: cough_db/eabb0dbf-be86-4c8d-922c-8bbc7f6557f1.webm  
  inflating: cough_db/eac1a08a-7388-4e0a-9fa7-24bd648ac15a.json  
  inflating: cough_db/eac1a08a-7388-4e0a-9fa7-24bd648ac15a.ogg  
  inflating: cough_db/eac1be19-f525-4b99-9a11-bec7122e4a66.json  
  inflating: cough_db/eac1be19-f525-4b99-9a11-bec7122e4a66.webm  
  inflating: cough_db/eac736db-4898-4ddb-b4fe-d4d5b7128af1.json  
  inflating: cough_db/eac736db-4898-4ddb-b4fe-d4d5b7128af1.webm  
  inflating: cough_db/eaccb237-9692-47e1-8401-cfcebc5a255b.json  
  inflating: cough_db/eaccb237-9692-47e1-8401-cfcebc5a255b.webm  
  inflating: cough_db/eace5f0f-09bc-42d9-a46b-01cbe774ce5c.json  
  inflating: cough_db/eace5f0f-09bc-42d9-a46b-01cbe774ce5c.webm  
  inflating: cough_db/ead1e5c6-26b6-4ea8-a1e1-2a01fac69359.json  
  inflating: cough_db/ead1

In [4]:
# ============================================================
# CELL 3 — Imports
# ============================================================
import os, torch, librosa, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from transformers import pipeline
import torchaudio

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [5]:
# ============================================================
# CELL 4 — Feature extraction (rich, consistent)
# ============================================================
def extract_features(path, sr=16000, duration=5):
    try:
        y, orig_sr = librosa.load(path, sr=sr, duration=duration)
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

    # Pad if shorter than duration
    target_len = sr * duration
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))

    feats = []

    # MFCCs (40 coefficients × mean + std = 80)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    feats.extend(np.mean(mfcc, axis=1))
    feats.extend(np.std(mfcc, axis=1))

    # Delta MFCCs (captures rate of change)
    delta_mfcc = librosa.feature.delta(mfcc)
    feats.extend(np.mean(delta_mfcc, axis=1))

    # Log-Mel spectrogram stats
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    log_mel = librosa.power_to_db(mel)
    feats.append(np.mean(log_mel))
    feats.append(np.std(log_mel))

    # Spectral features
    feats.append(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feats.append(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
    feats.append(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    feats.append(np.mean(librosa.feature.zero_crossing_rate(y)))

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    feats.extend(np.mean(chroma, axis=1))

    # RMS energy
    feats.append(np.mean(librosa.feature.rms(y=y)))

    return np.array(feats, dtype=np.float32)

In [6]:
# ============================================================
# CELL 5 — Build dataset from ICBHI Respiratory Sound DB
# Labels: normal / crackle / wheeze / both
# ============================================================
RESP_AUDIO = "respiratory_db/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files"

records = []
for txt_file in Path(RESP_AUDIO).glob("*.txt"):
    wav_file = txt_file.with_suffix(".wav")
    if not wav_file.exists():
        continue
    with open(txt_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            crackle = int(parts[2])
            wheeze  = int(parts[3])
            if crackle and wheeze:
                label = "both"
            elif crackle:
                label = "crackle"
            elif wheeze:
                label = "wheeze"
            else:
                label = "normal"
            records.append({"path": str(wav_file), "label": label})

resp_df = pd.DataFrame(records)
print("ICBHI dataset:")
print(resp_df["label"].value_counts())

ICBHI dataset:
label
normal     3642
crackle    1864
wheeze      886
both        506
Name: count, dtype: int64


In [14]:
# ============================================================
# CELL 6 — Build dataset from COVID Cough DB
# Labels: COVID-19 / healthy / symptomatic
# ============================================================
COUGH_META = "cough_db/metadata_compiled.csv"
COUGH_AUDIO = "cough_db/"

cough_meta = pd.read_csv(COUGH_META)
print(cough_meta.columns.tolist())
print(cough_meta.head(2))
cough_records=[]
for _, row in cough_meta.iterrows():
    status = str(row.get("status", "nan")).strip().lower()

    # Skip NaN/unknown labels
    if status in ["nan", "none", "", "unknown"]:
        continue

    uuid = str(row["uuid"])

    # Try both .ogg and .webm
    audio_path = None
    for ext in [".ogg", ".webm"]:
        path = os.path.join("cough_db", uuid + ext)
        if os.path.exists(path):
            audio_path = path
            break

    if audio_path:
        # Simplify label: covid_19 → covid, symptomatic → symptomatic, healthy → healthy
        label = "cough_" + status.replace("-", "_").replace(" ", "_")
        cough_records.append({"path": audio_path, "label": label})

cough_df = pd.DataFrame(cough_records)
print("Cough dataset (clean):")
print(cough_df["label"].value_counts())
print(f"Total: {len(cough_df)} samples")

['uuid', 'datetime', 'cough_detected', 'SNR', 'latitude', 'longitude', 'age', 'gender', 'respiratory_condition', 'fever_muscle_pain', 'status', 'quality_1', 'cough_type_1', 'dyspnea_1', 'wheezing_1', 'stridor_1', 'choking_1', 'congestion_1', 'nothing_1', 'diagnosis_1', 'severity_1', 'quality_2', 'cough_type_2', 'dyspnea_2', 'wheezing_2', 'stridor_2', 'choking_2', 'congestion_2', 'nothing_2', 'diagnosis_2', 'severity_2', 'quality_3', 'cough_type_3', 'dyspnea_3', 'wheezing_3', 'stridor_3', 'choking_3', 'congestion_3', 'nothing_3', 'diagnosis_3', 'severity_3', 'quality_4', 'cough_type_4', 'dyspnea_4', 'wheezing_4', 'stridor_4', 'choking_4', 'congestion_4', 'nothing_4', 'diagnosis_4', 'severity_4']
                                   uuid                          datetime  \
0  00014dcc-0f06-4c27-8c7b-737b18a2cf4c  2020-11-25T18:58:50.488301+00:00   
1  00039425-7f3a-42aa-ac13-834aaa2b6b92  2020-04-13T21:30:59.801831+00:00   

   cough_detected        SNR  latitude  longitude   age gender  

In [15]:
# ── UPDATED CELL 7 — Combine all 3 datasets ───────────────────────────────
import re
from pathlib import Path

# --- Lung dataset (parse labels from filename) ---
lung_records = []
label_map = {
    "asthma": "asthma",
    "copd": "copd",
    "heart failure": "heart_failure",
    "pneumonia": "pneumonia",
    "lung fibrosis": "lung_fibrosis",
    "bron": "bronchitis",
    "plueral effusion": "pleural_effusion",
}

for wav in Path("lung_db/Audio Files").glob("*.wav"):
    fname = wav.stem.lower()
    label = None
    for key, val in label_map.items():
        if key in fname:
            label = val
            break
    if label is None and re.search(r'_n,n,', fname):
        label = "normal"
    if label:
        lung_records.append({"path": str(wav), "label": label})

lung_df = pd.DataFrame(lung_records)
print("Lung dataset:")
print(lung_df["label"].value_counts())

# --- Combine all 3 ---
all_df = pd.concat([resp_df, cough_df, lung_df], ignore_index=True)
print(f"\nCombined total: {len(all_df)} samples")
print(all_df["label"].value_counts())

Lung dataset:
label
normal              105
asthma               99
heart_failure        57
copd                 33
pneumonia            15
lung_fibrosis        12
bronchitis            9
pleural_effusion      6
Name: count, dtype: int64

Combined total: 23458 samples
label
cough_healthy        12479
normal                3747
cough_symptomatic     2590
crackle               1864
cough_covid_19        1155
wheeze                 886
both                   506
asthma                  99
heart_failure           57
copd                    33
pneumonia               15
lung_fibrosis           12
bronchitis               9
pleural_effusion         6
Name: count, dtype: int64


In [16]:
# ── CELL 8 — Feature extraction with progress bar ─────────────────────────
from tqdm import tqdm

# Cap cough_healthy to 3000 to avoid heavy class imbalance
healthy_mask = all_df["label"] == "cough_healthy"
df_healthy = all_df[healthy_mask].sample(3000, random_state=42)
df_rest = all_df[~healthy_mask]
df_balanced = pd.concat([df_healthy, df_rest], ignore_index=True).sample(frac=1, random_state=42)

print(f"Balanced dataset: {len(df_balanced)} samples")
print(df_balanced["label"].value_counts())

X_list, y_list, failed = [], [], 0

for _, row in tqdm(df_balanced.iterrows(), total=len(df_balanced), desc="Extracting features"):
    feats = extract_features(row["path"])
    if feats is not None:
        X_list.append(feats)
        y_list.append(row["label"])
    else:
        failed += 1

print(f"\nDone! Extracted {len(X_list)} samples, {failed} failed")
X = np.array(X_list)
y = np.array(y_list)
print(f"Feature matrix: {X.shape}")

Balanced dataset: 13979 samples
label
normal               3747
cough_healthy        3000
cough_symptomatic    2590
crackle              1864
cough_covid_19       1155
wheeze                886
both                  506
asthma                 99
heart_failure          57
copd                   33
pneumonia              15
lung_fibrosis          12
bronchitis              9
pleural_effusion        6
Name: count, dtype: int64


Extracting features: 100%|██████████| 13979/13979 [37:36<00:00,  6.20it/s]



Done! Extracted 13979 samples, 0 failed
Feature matrix: (13979, 139)


In [38]:
# ============================================================
# CELL 8 — Clean dataset, merge into 3 classes, scale, train XGB & RF, ensemble
# ============================================================

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

# ── Step 1: Convert y to pandas Series
y = pd.Series(y)

# ── Step 2: Merge classes into 3 final classes
y = y.replace({
    # Severe cough
    'cough_covid_19': 'cough_severe',
    'cough_symptomatic': 'cough_severe',
    'cough_infected': 'cough_severe',
    'bronchitis': 'cough_severe',
    'pneumonia': 'cough_severe',
    'lung_fibrosis': 'cough_severe',
    'pleural_effusion': 'cough_severe',
    # Healthy cough
    'cough_healthy': 'cough_healthy',
    # All others → other
    'asthma': 'other',
    'both': 'other',
    'crackle': 'other',
    'normal': 'other',
    'wheeze': 'other'
})

print("Final classes:", sorted(y.unique()))

# ── Step 3: Convert X to numpy array
X = np.array(X)

# ── Step 4: Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Encoded classes:", le.classes_)

# ── Step 5: Split dataset
X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# ── Step 6: Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

# ── Step 7: Compute sample weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
sample_weights = np.array([class_weights[i] for i in y_train])

# ── Step 8: Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    reg_alpha=0.5,
    objective="multi:softprob",
    num_class=len(classes),
    eval_metric="mlogloss",
    tree_method="hist",
    device=device,
    early_stopping_rounds=30,
    random_state=42
)
xgb_model.fit(
    X_train_s, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_s, y_val)],
    verbose=50
)

# ── Step 9: Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_s, y_train, sample_weight=sample_weights)

# ── Step 10: Predict probabilities
xgb_probs = xgb_model.predict_proba(X_val_s)
rf_probs  = rf_model.predict_proba(X_val_s)

# ── Step 11: Ensemble (average probabilities)
final_preds = np.argmax((xgb_probs + rf_probs)/2, axis=1)

# ── Step 12: Evaluate models
xgb_acc = accuracy_score(y_val, xgb_model.predict(X_val_s))
rf_acc  = accuracy_score(y_val, rf_model.predict(X_val_s))
ensemble_acc = accuracy_score(y_val, final_preds)

print(f"\nXGBoost accuracy: {xgb_acc:.4f}")
print(f"Random Forest accuracy: {rf_acc:.4f}")
print(f"Ensemble accuracy: {ensemble_acc:.4f}")

# ── Step 13: Select best individual model
best_model = xgb_model if xgb_acc >= rf_acc else rf_model
best_name  = "XGBoost" if xgb_acc >= rf_acc else "RandomForest"
print(f"\nBest model: {best_name}")

# ── Step 14: Classification report
print(classification_report(
    y_val,
    best_model.predict(X_val_s),
    target_names=le.classes_
))

Final classes: ['cough_healthy', 'cough_severe', 'other']
Encoded classes: ['cough_healthy' 'cough_severe' 'other']
[0]	validation_0-mlogloss:1.05221
[50]	validation_0-mlogloss:0.40541
[100]	validation_0-mlogloss:0.34919
[150]	validation_0-mlogloss:0.34114
[200]	validation_0-mlogloss:0.34041
[222]	validation_0-mlogloss:0.34080

XGBoost accuracy: 0.7780
Random Forest accuracy: 0.7877
Ensemble accuracy: 0.7834

Best model: RandomForest
               precision    recall  f1-score   support

cough_healthy       0.53      0.19      0.28       600
 cough_severe       0.57      0.86      0.69       749
        other       1.00      1.00      1.00      1421

     accuracy                           0.79      2770
    macro avg       0.70      0.68      0.66      2770
 weighted avg       0.78      0.79      0.76      2770



In [40]:
# ============================================================
# CELL 10 — STT with Whisper (unchanged, already good)
# ============================================================
stt = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if device == "cuda" else -1
)

def transcribe_audio(path):
    result = stt(path)
    return result["text"]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [41]:
# ============================================================
# CELL 12 — Full pipeline (replaces old hardcoded version)
# ============================================================
def run_audio_pipeline(path):
    feats = extract_features(path)
    if feats is None:
        return {"error": "Could not load audio"}

    feats_scaled = scaler.transform(feats.reshape(1, -1))
    probs = best_model.predict_proba(feats_scaled)[0]
    top3  = np.argsort(probs)[-3:][::-1]

    predictions = [{
        "label":      le.inverse_transform([i])[0],
        "confidence": round(float(probs[i]), 4),
        "confidence_label": "High" if probs[i] > 0.6 else "Medium" if probs[i] > 0.3 else "Low"
    } for i in top3]

    return {
        "top_prediction": predictions[0],
        "all_predictions": predictions,
        "transcription": transcribe_audio(path)
    }

In [42]:
# ============================================================
# CELL 13 — Save everything
# ============================================================
joblib.dump(best_model, "audio_model.pkl")
joblib.dump(scaler,     "audio_scaler.pkl")
joblib.dump(le,         "audio_label_encoder.pkl")
print("✅ Audio model saved!")

✅ Audio model saved!
